# Sensor-Daten als Graph modellieren und nach Neo4j exportieren

In diesem Notebook bauen wir von Grund auf eine sinnvolle Graph-Repräsentation für einen Datensatz mit 54 Sensoren, die in Epochen Messwerte liefern. Wir zeigen Schritt für Schritt, wie man:

- die Daten lädt und inspiziert
- ein geeignetes Graph-Modell entwirft
- den Graphen mit NetworkX erstellt
- und das Ergebnis nach Neo4j exportiert.

Jeder Schritt ist einzeln nachvollziehbar und eignet sich gut zur Präsentation.

## 1. Daten laden und inspizieren

Wir laden die CSV-Datei und werfen einen Blick auf die Struktur und die wichtigsten Eigenschaften des Datensatzes.

In [17]:
import os
import pandas as pd

CSV_PATH = "data/graph/data.csv"

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV nicht gefunden: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
print(f"Anzahl Zeilen: {len(df)}")
print(f"Spalten: {df.columns.tolist()}")
print(f"Anzahl Sensoren (moteid): {df['moteid'].nunique()}")
df.head()

Anzahl Zeilen: 465
Spalten: ['date', 'time', 'epoch', 'moteid', 'temperature', 'humidity', 'light', 'voltage']
Anzahl Sensoren (moteid): 54


,date,time,epoch,moteid,temperature,humidity,light,voltage
0,2004-03-31,03:38:15.757551,2,1,122.1530,-3.91901,11.04,2.03397
1,2004-02-28,00:59:16.02785,3,1,19.9884,37.09330,45.08,2.69964
2,2004-02-28,01:03:16.33393,11,1,19.3024,38.46290,45.08,2.68742
3,2004-02-28,01:06:16.013453,17,1,19.1652,38.80390,45.08,2.68742
4,2004-02-28,01:06:46.778088,18,1,19.1750,38.83790,45.08,2.69964


## 2. Graph-Modellierung

Wir wählen folgende Struktur:

- **Sensor**-Knoten (`Sensor`): Jeder Sensor (moteid) ist ein Knoten.
- **Epoch**-Knoten (`Epoch`): Jede Epoche ist ein Knoten.
- **Measurement**-Knoten (`Measurement`): Jede einzelne Messung ist ein Knoten mit den Messwerten.

Beziehungen:

- `(:Measurement)-[:MEASURED_BY]->(:Sensor)`
- `(:Measurement)-[:MEASURED_IN]->(:Epoch)`

So können wir flexibel nach Sensoren, Zeitpunkten oder Messwerten suchen.

## 3. Graph mit NetworkX erstellen

Wir bauen den Graphen entsprechend des Modells auf.

In [18]:
import networkx as nx

G = nx.DiGraph()

# Sensor-Knoten anlegen (ein Knoten pro Sensor)
for moteid in df['moteid'].unique():
    G.add_node(f"sensor_{moteid}", label="Sensor", moteid=int(moteid))

# Epoch-Knoten anlegen (ein Knoten pro Epoche)
for epoch in df['epoch'].unique():
    G.add_node(f"epoch_{epoch}", label="Epoch", epoch=int(epoch))

# Measurement-Knoten und Kanten anlegen
for idx, row in df.iterrows():
    meas_id = f"meas_{idx}"
    sensor_id = f"sensor_{int(row['moteid'])}"
    epoch_id = f"epoch_{int(row['epoch'])}"
    # Messungsknoten mit Messwerten und Referenzen
    G.add_node(
        meas_id,
        label="Measurement",
        temperature=row['temperature'],
        humidity=row['humidity'],
        light=row['light'],
        voltage=row['voltage'],
        date=row.get('date'),
        time=row.get('time'),
        moteid=int(row['moteid']),
        epoch=int(row['epoch'])
    )
    # Beziehung zur Sensor- und Epoch-Knoten
    G.add_edge(meas_id, sensor_id, type="MEASURED_BY")
    G.add_edge(meas_id, epoch_id, type="MEASURED_IN")

print(f"Graph: {G.number_of_nodes()} Knoten, {G.number_of_edges()} Kanten")

Graph: 539 Knoten, 930 Kanten


## 4. Export nach Neo4j

Wir exportieren den Graphen nach Neo4j. Falls Neo4j nicht erreichbar ist, speichern wir den Graphen lokal als GraphML.

In [19]:
from neo4j import GraphDatabase

NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://neo4j:7687")
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "password")

def export_to_neo4j(G, uri, user, password, wipe=True, batch_size=1000):
    driver = GraphDatabase.driver(uri, auth=(user, password))
    try:
        with driver.session() as session:
            if wipe:
                session.run("MATCH (n) DETACH DELETE n")
                print("✓ Neo4j: vorhandene Daten gelöscht")

            # Knoten nach Label gruppieren
            sensors = [ (n, d) for n, d in G.nodes(data=True) if d.get('label') == 'Sensor']
            epochs = [ (n, d) for n, d in G.nodes(data=True) if d.get('label') == 'Epoch']
            measurements = [ (n, d) for n, d in G.nodes(data=True) if d.get('label') == 'Measurement']

            def chunked(lst, n):
                for i in range(0, len(lst), n):
                    yield lst[i:i+n]

            # Sensor-Knoten
            for batch in chunked(sensors, batch_size):
                session.run("""
                    UNWIND $batch AS row
                    MERGE (s:Sensor {uid: row[0]})
                    SET s += row[1]
                """, batch=batch)

            # Epoch-Knoten
            for batch in chunked(epochs, batch_size):
                session.run("""
                    UNWIND $batch AS row
                    MERGE (e:Epoch {uid: row[0]})
                    SET e += row[1]
                """, batch=batch)

            # Measurement-Knoten
            for batch in chunked(measurements, batch_size):
                session.run("""
                    UNWIND $batch AS row
                    MERGE (m:Measurement {uid: row[0]})
                    SET m += row[1]
                """, batch=batch)

            # Kanten
            rels = [ (u, v, d) for u, v, d in G.edges(data=True)]
            for batch in chunked(rels, batch_size):
                session.run("""
                    UNWIND $batch AS row
                    MATCH (src {uid: row[0]}), (dst {uid: row[1]})
                    MERGE (src)-[r:REL {type: row[2]['type']}]->(dst)
                """, batch=batch)

        print("✓ Export nach Neo4j abgeschlossen")
    finally:
        driver.close()

try:
    export_to_neo4j(G, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
except Exception as e:
    print("Neo4j nicht erreichbar – speichere Graph lokal als graph.graphml:", e)
    os.makedirs("data/graph/out", exist_ok=True)
    nx.write_graphml(G, "data/graph/out/graph.graphml")
    print("✓ Graph lokal gespeichert: data/graph/out/graph.graphml")

✓ Neo4j: vorhandene Daten gelöscht
✓ Export nach Neo4j abgeschlossen
✓ Export nach Neo4j abgeschlossen


## 5. Beispielhafte Cypher-Abfragen für die Präsentation in Neo4j

Hier sind einige spannende Cypher-Queries, mit denen du den Graphen in Neo4j präsentieren kannst:

```cypher
// 1. Alle Sensoren anzeigen
MATCH (s:Sensor) RETURN s LIMIT 10;

// 2. Alle Epochen anzeigen
MATCH (e:Epoch) RETURN e LIMIT 10;

// 3. Alle Messungen eines bestimmten Sensors (z.B. Sensor 1)
MATCH (s:Sensor {moteid: 1})<-[:MEASURED_BY]-(m:Measurement)
RETURN m LIMIT 10;

// 4. Durchschnittstemperatur pro Sensor berechnen
MATCH (s:Sensor)<-[:MEASURED_BY]-(m:Measurement)
RETURN s.moteid AS Sensor, avg(m.temperature) AS Durchschnittstemperatur
ORDER BY Durchschnittstemperatur DESC;

// 5. Messungen in einer bestimmten Epoche (z.B. epoch=100)
MATCH (e:Epoch {epoch: 100})<-[:MEASURED_IN]-(m:Measurement)
RETURN m LIMIT 10;

// 6. Sensoren mit den meisten Messungen
MATCH (s:Sensor)<-[:MEASURED_BY]-(m:Measurement)
RETURN s.moteid AS Sensor, count(m) AS Messungen
ORDER BY Messungen DESC
LIMIT 10;

// 7. Zeitlicher Verlauf der Temperatur eines Sensors
MATCH (s:Sensor {moteid: 1})<-[:MEASURED_BY]-(m:Measurement)
RETURN m.epoch AS Epoche, m.temperature AS Temperatur
ORDER BY Epoche ASC;

// 8. Messungen mit ungewöhnlich hoher Temperatur (>30°C)
MATCH (m:Measurement)
WHERE m.temperature > 30
RETURN m LIMIT 10;
```

Du kannst diese Abfragen direkt in Neo4j ausführen, um verschiedene Aspekte des Graphen zu präsentieren!
```